### 뉴스 데이터 1000 개
- 본문을 읽고 정형 데이터로 변환하는 자동화 파이프 라인 만들기
- 1000개 데이터의 내용을 카테고리, 핵심어, 요약하면 
- 항목
    - 카테고리
    - 요약
    - 핵심어
-구조화된 출력

### 파이프라인 설계
1. 읽기 : 뉴스 본문 불러오기
2. 분석 : 각 기사를 정해진 항목으로 분석
3. 정리,저장  : 표로 모아서 csv저장

In [1]:
import pandas as pd
df = pd.read_csv("../data/11-1_뉴스정제.csv")
df = df.head(5)

In [2]:
import json
import os
from datetime import datetime

In [4]:
#주식관련
def get_stock_price(stock_name):
    stock_data = {
        "삼성전자": {
            "price": 70000,
            "change": "+1.2%"
        },
        "애플": {
            "price": 230,
            "change": "-0.5%"
        },
        "테슬라": {
            "price": 350,
            "change": "+2.1%"
        }
    }

    if stock_name not in stock_data:
        return f"{stock_name}의 주가 정보를 찾을 수 없습니다."

    data = stock_data[stock_name]

    return (
        f"{stock_name} 현재 주가: {data['price']:,}\n"
        f"등락률: {data['change']}\n"
        f"※ 학습용 예시 데이터입니다."
    )


In [3]:
def save_result(filename, question, answer):
      content = f"""# 질문과 답변

  ## 질문
  {question}

  ## 답변
  {answer}

  ## 저장 시간
  {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
  """

      with open(filename, "w", encoding="utf-8-sig") as file:
          file.write(content)

      return f"{filename} 파일에 저장했습니다."

In [5]:
def save_result(filename, question, answer):
    content = f"""# 질문과 답변

## 질문
{question}

## 답변
{answer}

## 저장 시간
{datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""

    with open(filename, "w", encoding="utf-8-sig") as file:
        file.write(content)

    return f"{filename} 파일에 저장했습니다."


def read_result(filename):
    if not os.path.exists(filename):
        return f"{filename} 파일이 존재하지 않습니다."

    with open(filename, "r", encoding="utf-8-sig") as file:
        return file.read()

In [6]:
available_tools = {
    "recommend_cloth": recommend_cloth,
    "check_data": check_data,
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "save_result": save_result,
    "read_result": read_result
}

NameError: name 'recommend_cloth' is not defined

In [ ]:
def chat_with_tools(question, tools):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    response = client.chat.completions.create(
        model="gpt-5.6-luna",
        tools=tools,
        reasoning_effort="none",
        messages=messages
    )

    calls = response.choices[0].message.tool_calls

    if not calls:
        return response.choices[0].message.content

    messages.append(response.choices[0].message)

    for tc in calls:
        args = json.loads(tc.function.arguments)

        tool_function = available_tools[tc.function.name]
        tool_result = tool_function(**args)

        messages.append({
            "role": "tool",
            "tool_call_id": tc.id,
            "content": str(tool_result)
        })

    final_response = client.chat.completions.create(
        model="gpt-5.6-luna",
        tools=tools,
        reasoning_effort="none",
        messages=messages
    )

    return final_response.choices[0].message.content

In [ ]:
print(chat_with_tools("삼성전자 주가 알려줘", tools))
print(chat_with_tools("result.md 파일 내용 보여줘", tools))

messages = [
    {
        "role": "user",
        "content": "신대방 날씨 어때?"
    }
]